# Yield Curve Decoded

PCA decomposition of the US Treasury yield curve and recession prediction from curve shape features.

**Data:** FRED API - 8 Treasury constant maturity series (1M through 30Y) + NBER recession indicator, daily from 1976 to present.

**Methods:** Principal Component Analysis (level, slope, curvature) + XGBoost recession classifier with SHAP interpretability.

In [ ]:
import pandas as pd
import numpy as np
import requests
import os
import time
from datetime import datetime, timedelta
import xgboost as xgb
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import (
    roc_auc_score, precision_recall_curve, auc,
    confusion_matrix, classification_report, roc_curve
)
from sklearn.model_selection import ParameterSampler
import shap
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.dates as mdates
import seaborn as sns
import plotly.graph_objects as go
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", 30)

In [ ]:
C = {
    "primary":    "#1e293b",
    "secondary":  "#334155",
    "accent":     "#2563eb",
    "risk":       "#dc2626",
    "safe":       "#059669",
    "warn":       "#d97706",
    "recession":  "#fee2e2",
    "light_gray": "#f1f5f9",
    "mid_gray":   "#94a3b8",
    "dark_gray":  "#475569",
}

plt.rcParams.update({
    "figure.facecolor": "#ffffff",
    "axes.facecolor": "#ffffff",
    "axes.edgecolor": "#e2e8f0",
    "axes.labelcolor": C["primary"],
    "axes.titlecolor": C["primary"],
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "axes.grid": True,
    "grid.color": "#f1f5f9",
    "grid.linewidth": 0.8,
    "xtick.color": C["dark_gray"],
    "ytick.color": C["dark_gray"],
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "font.family": "sans-serif",
    "font.sans-serif": ["Segoe UI", "Helvetica Neue", "Arial"],
    "figure.dpi": 100,
    "legend.frameon": False,
    "legend.fontsize": 9,
})

def style_axis(ax, title="", subtitle="", xlabel="", ylabel=""):
    if title:
        ax.set_title(title, fontsize=14, fontweight="bold", color=C["primary"], pad=12, loc="left")
    if subtitle:
        ax.text(0, 1.02, subtitle, transform=ax.transAxes, fontsize=10,
                color=C["dark_gray"], style="italic", va="bottom")
    if xlabel:
        ax.set_xlabel(xlabel, fontsize=11, color=C["secondary"])
    if ylabel:
        ax.set_ylabel(ylabel, fontsize=11, color=C["secondary"])
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color("#e2e8f0")
    ax.spines["bottom"].set_color("#e2e8f0")
    return ax

def shade_recessions(ax, recessions_series):
    """Add light red shading for NBER recession periods."""
    in_recession = False
    start = None
    for date, val in recessions_series.items():
        if val == 1 and not in_recession:
            start = date
            in_recession = True
        elif val == 0 and in_recession:
            ax.axvspan(start, date, alpha=0.15, color=C["risk"], zorder=0)
            in_recession = False
    if in_recession:
        ax.axvspan(start, recessions_series.index[-1], alpha=0.15, color=C["risk"], zorder=0)

def fmt_k(x, _=None):
    if abs(x) >= 1e3:
        return f"{x/1e3:.0f}K"
    return f"{x:.0f}"

print("Style configured.")

In [ ]:
FRED_API_KEY = os.environ.get("FRED_API_KEY", "PASTE_YOUR_KEY_HERE")
FRED_BASE_URL = "https://api.stlouisfed.org/fred/series/observations"

def fetch_fred(series_id, start_date="1976-01-01", max_retries=3):
    """Fetch a FRED series and return as pd.Series indexed by date."""
    params = {
        "series_id": series_id,
        "api_key": FRED_API_KEY,
        "file_type": "json",
        "observation_start": start_date,
    }
    for attempt in range(max_retries):
        try:
            resp = requests.get(FRED_BASE_URL, params=params, timeout=30)
            if resp.status_code == 429:
                time.sleep(2 ** attempt)
                continue
            resp.raise_for_status()
            data = resp.json()
            break
        except requests.exceptions.RequestException as exc:
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
                continue
            raise RuntimeError(f"Failed to fetch {series_id}: {exc}")

    df = pd.DataFrame(data["observations"])
    df["date"] = pd.to_datetime(df["date"])
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    s = df.set_index("date")["value"].dropna()
    s.name = series_id
    return s

print("FRED fetcher ready.")

In [ ]:
YIELD_SERIES = {
    "1M":  "DGS1MO",
    "3M":  "DGS3MO",
    "6M":  "DGS6MO",
    "1Y":  "DGS1",
    "2Y":  "DGS2",
    "5Y":  "DGS5",
    "10Y": "DGS10",
    "30Y": "DGS30",
}

MATURITY_ORDER = ["1M", "3M", "6M", "1Y", "2Y", "5Y", "10Y", "30Y"]
MATURITY_YEARS = [1/12, 3/12, 6/12, 1, 2, 5, 10, 30]

print("Fetching Treasury yields...")
yields = {}
for label, sid in YIELD_SERIES.items():
    yields[label] = fetch_fred(sid)
    print(f"  {label} ({sid}): {len(yields[label]):,} obs")

curve_df = pd.DataFrame(yields).sort_index()
curve_df = curve_df.dropna()

print(f"\nFetching NBER recession indicator...")
recession = fetch_fred("USREC", start_date="1976-01-01")
recession_daily = recession.reindex(curve_df.index, method="ffill").fillna(0).astype(int)

print(f"\nYield curve matrix: {curve_df.shape[0]:,} days x {curve_df.shape[1]} maturities")
print(f"Date range: {curve_df.index.min():%Y-%m-%d} to {curve_df.index.max():%Y-%m-%d}")
print(f"Recession days: {recession_daily.sum():,} ({recession_daily.mean():.1%})")

---
## Exploratory Data Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

current = curve_df.iloc[-1]
ax.plot(MATURITY_YEARS, current.values, "o-", lw=3, color=C["accent"],
        markersize=8, label=f"Current ({curve_df.index[-1]:%Y-%m-%d})", zorder=5)

recession_mask = recession_daily.reindex(curve_df.index).fillna(0) == 1
if recession_mask.sum() > 0:
    recession_avg = curve_df[recession_mask].mean()
    ax.plot(MATURITY_YEARS, recession_avg.values, "s--", lw=2, color=C["risk"],
            markersize=6, label="Avg during recessions")

expansion_avg = curve_df[~recession_mask].mean()
ax.plot(MATURITY_YEARS, expansion_avg.values, "^--", lw=2, color=C["safe"],
        markersize=6, label="Avg during expansions")

ax.set_xscale("log")
ax.set_xticks(MATURITY_YEARS)
ax.set_xticklabels(MATURITY_ORDER)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0, decimals=1))
ax.legend()
style_axis(ax, title="US Treasury Yield Curve: Current vs Historical",
           subtitle="How does today's curve compare to recession and expansion averages?",
           xlabel="Maturity", ylabel="Yield (%)")
plt.tight_layout()
plt.show()

In [ ]:
spread_2s10s = curve_df["10Y"] - curve_df["2Y"]

fig, ax = plt.subplots(figsize=(16, 5))
shade_recessions(ax, recession_daily)
ax.plot(spread_2s10s.index, spread_2s10s.values, lw=1, color=C["accent"])
ax.axhline(0, color=C["risk"], lw=1.5, ls="--", alpha=0.7)
ax.fill_between(spread_2s10s.index, spread_2s10s.values, 0,
                where=spread_2s10s < 0, alpha=0.3, color=C["risk"], label="Inverted")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0, decimals=1))
ax.legend()
style_axis(ax, title="2s10s Treasury Spread (10Y - 2Y)",
           subtitle="Red shading = NBER recession. Inversion has preceded every recession since 1976.",
           xlabel="", ylabel="Spread (%)")
plt.tight_layout()
plt.show()

In [ ]:
spread_3m10y = curve_df["10Y"] - curve_df["3M"]

fig, ax = plt.subplots(figsize=(16, 5))
shade_recessions(ax, recession_daily)
ax.plot(spread_3m10y.index, spread_3m10y.values, lw=1, color=C["safe"])
ax.axhline(0, color=C["risk"], lw=1.5, ls="--", alpha=0.7)
ax.fill_between(spread_3m10y.index, spread_3m10y.values, 0,
                where=spread_3m10y < 0, alpha=0.3, color=C["risk"], label="Inverted")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0, decimals=1))
ax.legend()
style_axis(ax, title="3m10Y Treasury Spread (10Y - 3M)",
           subtitle="Often cited as the stronger recession predictor. Sharper inversions than 2s10s.",
           xlabel="", ylabel="Spread (%)")
plt.tight_layout()
plt.show()

In [ ]:
yearly_avg = curve_df.groupby(curve_df.index.year).mean()
yearly_avg = yearly_avg[MATURITY_ORDER]

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(yearly_avg, cmap="YlOrRd", annot=True, fmt=".1f", ax=ax,
            linewidths=0.3, linecolor="#e2e8f0",
            cbar_kws={"label": "Yield (%)"})
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
style_axis(ax, title="Average Yield by Year and Maturity",
           subtitle="The secular decline in rates from Volcker-era highs to the post-GFC floor",
           xlabel="Maturity", ylabel="Year")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
data_list = [curve_df[m].dropna().values for m in MATURITY_ORDER]
parts = ax.violinplot(data_list, positions=range(len(MATURITY_ORDER)), showmedians=True, showextrema=False)
for pc in parts["bodies"]:
    pc.set_facecolor(C["accent"])
    pc.set_alpha(0.4)
ax.set_xticks(range(len(MATURITY_ORDER)))
ax.set_xticklabels(MATURITY_ORDER)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0, decimals=0))
style_axis(ax, title="Yield Distribution by Maturity (1976-Present)",
           subtitle="Full historical range. Short rates show more dispersion due to Fed policy cycles.",
           xlabel="Maturity", ylabel="Yield (%)")
plt.tight_layout()
plt.show()

In [ ]:
curve_sampled = curve_df.iloc[::5]

fig = go.Figure(data=[go.Surface(
    z=curve_sampled.values,
    x=MATURITY_YEARS,
    y=curve_sampled.index,
    colorscale="YlOrRd",
    colorbar=dict(title="Yield (%)"),
)])
fig.update_layout(
    title=dict(text="US Treasury Yield Curve Surface (1976-Present)", font=dict(size=16, color="#1e293b")),
    scene=dict(
        xaxis_title="Maturity (Years)",
        yaxis_title="Date",
        zaxis_title="Yield (%)",
        xaxis=dict(type="log"),
    ),
    paper_bgcolor="#ffffff",
    width=900, height=600,
    margin=dict(l=0, r=0, t=50, b=0),
)
fig.show()